In [0]:

# Databricks notebook source
# Download taxi_zone_lookup.csv from https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

import requests
import os
import io

# Destination dans DBFS
DBFS_FOLDER = "dbfs:/raw_data_files/"
FILE_NAME = "taxi_zone_lookup.csv"
URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Chemin DBFS temporaire
DBFS_TMP = f"{DBFS_FOLDER}{FILE_NAME}"

def download_taxi_zone_lookup():
    """Téléchargez le fichier taxi_zone_lookup.csv et enregistrez-le dans le DBFS."""
    dbfs_path = os.path.join(DBFS_FOLDER, FILE_NAME)

    # Ignorer si le fichier existe déjà
    from pyspark.dbutils import DBUtils
    dbutils = DBUtils(spark)
    try:
        dbutils.fs.ls(dbfs_path)
        print(f"✅ {FILE_NAME} existe déjà dans le DBFS, étape ignorée...")
        return
    except Exception:
        pass

    print(f"⬇️ Téléchargement en cours {FILE_NAME}...")
    response = requests.get(URL, stream=True)

    if response.status_code == 200:
        # Verifier que response.content est correctement décodé en chaîne de caractères.
        decoded_content = response.content.decode('utf-8')
        # Enregistrer directement dans le DBFS
        dbutils.fs.put(DBFS_TMP, decoded_content, overwrite=True)
        print(f"✅ Enregistré {FILE_NAME} vers {dbfs_path}")
    else:
        print(f"❌ Échec du téléchargement {FILE_NAME}. HTTP status {response.status_code}")

# Run
download_taxi_zone_lookup()

✅ taxi_zone_lookup.csv existe déjà dans le DBFS, étape ignorée...


In [0]:


# Vérifiez si le fichier existe
dbutils.fs.ls("dbfs:/raw_data_files/taxi_zone_lookup.csv")

[FileInfo(path='dbfs:/raw_data_files/taxi_zone_lookup.csv', name='taxi_zone_lookup.csv', size=12331, modificationTime=1762933038000)]

In [0]:

# Téléchargement continue des fichiers parquets
import requests
from pyspark.dbutils import DBUtils
import tempfile
import os
from datetime import datetime, timedelta
import time

# Destination in DBFS
DBFS_FOLDER = "dbfs:/raw_data_files/"
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"

dbutils = DBUtils(spark)

def download_yellow_tripdata(year: int, month: int):
    """
   Télécharge un fichier Parquet pour une année-mois donnée si celui-ci n’existe pas déjà dans le DBFS.
    """
    file_name = f"yellow_tripdata_{year}-{month:02d}.parquet"
    dbfs_path = DBFS_FOLDER + file_name
    url = BASE_URL.format(year=year, month=month)

    # 1. Ignorer si le fichier existe déjà
    try:
        dbutils.fs.ls(dbfs_path)
        print(f"✅ {file_name} Existe déjà dans le DBFS, opération ignorée...")
        return
    except Exception:
        pass

    # 2. Créez un répertoire temporaire local sous Workspace
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        local_temp_dir = os.path.join(os.path.dirname("/Workspace" + notebook_path), ".tmp_downloads")
        os.makedirs(local_temp_dir, exist_ok=True)
    except Exception as e:
        print(f"❌ Impossible de créer un répertoire temporaire dans l’espace de travail. Erreur : {e}")
        return

    temp_file_path = None
    print(f"⬇️ Téléchargement en cours {file_name}...")
    response = requests.get(url, stream=True)

    if response.status_code == 200:
        try:
            with tempfile.NamedTemporaryFile(dir=local_temp_dir, delete=False, suffix=".parquet") as temp_file:
                temp_file_path = temp_file.name
                for chunk in response.iter_content(chunk_size=1024 * 1024):  # morceaux de 1 Mo
                    temp_file.write(chunk)

            # Copier dans le DBFS
            dbutils.fs.cp(f'file:{temp_file_path}', dbfs_path)
            print(f"✅ Enregistré {file_name} to {dbfs_path}")

        except Exception as e:
            print(f"❌ Échec de l’enregistrement du fichier {file_name} to DBFS: {e}")
        finally:
            if temp_file_path and os.path.exists(temp_file_path):
                os.remove(temp_file_path)
            try:
                if not os.listdir(local_temp_dir):
                    os.rmdir(local_temp_dir)
            except:
                pass
    else:
        print(f"❌ Échec du téléchargement {file_name}. HTTP status {response.status_code}")


# -------- BOUCLE PRINCIPALE --------
start_date = datetime(2025, 1, 1)
today = datetime.today()

year, month = start_date.year, start_date.month

while (year < today.year) or (year == today.year and month <= today.month):
    download_yellow_tripdata(year, month)

    # Passer au mois suivant
    if month == 12:
        year += 1
        month = 1
    else:
        month += 1

    # Pause de 5 minutes avant la prochaine itération
    print("⏸️ Attente de 5 minutes avant le prochain téléchargement...")
    time.sleep(5 * 60)

⬇️ Téléchargement en cours yellow_tripdata_2025-01.parquet...
✅ Enregistré yellow_tripdata_2025-01.parquet to dbfs:/raw_data_files/yellow_tripdata_2025-01.parquet
⏸️ Attente de 5 minutes avant le prochain téléchargement...
⬇️ Téléchargement en cours yellow_tripdata_2025-02.parquet...
✅ Enregistré yellow_tripdata_2025-02.parquet to dbfs:/raw_data_files/yellow_tripdata_2025-02.parquet
⏸️ Attente de 5 minutes avant le prochain téléchargement...
⬇️ Téléchargement en cours yellow_tripdata_2025-03.parquet...
✅ Enregistré yellow_tripdata_2025-03.parquet to dbfs:/raw_data_files/yellow_tripdata_2025-03.parquet
⏸️ Attente de 5 minutes avant le prochain téléchargement...
⬇️ Téléchargement en cours yellow_tripdata_2025-04.parquet...
✅ Enregistré yellow_tripdata_2025-04.parquet to dbfs:/raw_data_files/yellow_tripdata_2025-04.parquet
⏸️ Attente de 5 minutes avant le prochain téléchargement...
⬇️ Téléchargement en cours yellow_tripdata_2025-05.parquet...
✅ Enregistré yellow_tripdata_2025-05.parquet t